# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [196]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [220]:
from openai import OpenAI
import os

USE_GATEWAY = (os.getenv("USE_GATEWAY", "FALSE").upper() == "TRUE")
MODEL = os.getenv("MODEL", "gpt-4o-mini")

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:

    client = OpenAI(
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_key="any value",
        default_headers={
            "x-api-key": os.getenv("API_GATEWAY_KEY")
        }
    )

    return client

client = get_client()

In [221]:
import sys
sys.path.append('../../05_src/')

In [200]:
import os 
os.getenv("API_Gateway_key")

In [201]:
import requests

In [202]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [203]:
import pypdf
from langchain_community.document_loaders import PyPDFLoader

print("Success!")

Success!


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [204]:
file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

In [205]:
loader = PyPDFLoader(file_path)
pages = loader.load()

In [206]:
len(pages)
#number of pages in the PDF

26

In [207]:
pdf_text = "\n".join(page.page_content for page in pages)

In [208]:
print(docs[0].page_content)

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025


In [153]:
docs[0].metadata

{'producer': 'Microsoft® Word for Microsoft 365',
 'creator': 'Microsoft® Word for Microsoft 365',
 'creationdate': '2025-07-13T21:18:19-07:00',
 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47',
 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged',
 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True',
 'author': 'Aditya Challapally',
 'moddate': '2025-07-13T21:18:19-07:00',
 'source': 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf',
 'total_pages': 26,
 'page': 0,
 'page_label': '1'}

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [226]:

from pydantic import BaseModel, Field   
import os
os.environ["LANGSMITH_TRACING"] = "true"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()


In [217]:
class ArticleAnalysis(BaseModel):
    author: str = Field(
        description="The author of the article."
    )

    title: str = Field(
        description="The title of the article."
    )

    relevance: str = Field(
        description=(
            "A statement, no longer than one paragraph, explaining why "
            "the article is relevant to an AI professional's professional development."
        )
    )

    summary: str = Field(
        description=(
            "A concise and succinct summary of the article, "
            "no longer than 1000 tokens."
        )
    )

In [ ]:
developer_instructions = """
You are an AI professional reviewing professional and academic literature.

Analyze the supplied article and:
- Identify the author.
- Identify the title.
- Explain why the article is relevant to an AI professional's professional development.
- Keep the relevance statement to no more than one paragraph.
- Provide a concise and succinct summary of no more than 1000 tokens.
- Write the relevance statement and summary in legalese tone.

Base your analysis only on the supplied article.
"""

In [3]:
user_prompt = f"""
Analyze the following article:

<article>
{pdf_text}
</article>
"""

NameError: name 'pdf_text' is not defined

In [227]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "developer",
            "content": developer_instructions
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    text_format=ArticleAnalysis
)

PermissionDeniedError: Error code: 403 - {'message': 'Forbidden'}

In [219]:
article_analysis = response.output_parsed

AttributeError: 'Response' object has no attribute 'output_parsed'

In [224]:
print(article_analysis)

author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' title='The GenAI Divide: State of AI in Business 2025' relevance='This here article shines a light on the stark divide between high GenAI adoption and real transformation in business—figurin’ out what’s workin’ and what ain’t for folks in the AI world. For an AI professional, understanding these trends and patterns can help steer their ship toward more fruitful investments and implementations, keepin’ them ahead of the curve in a rapidly changin’ landscape.' summary='In the report "The GenAI Divide: State of AI in Business 2025", the authors reveal a tall tale about the state of Generative AI (GenAI) in the business frontier. Despite a mighty investment between $30–40 billion in GenAI initiatives, a staggering 95% of organizations are seeing no returns—what they call the GenAI Divide. The report outlines that while tools like ChatGPT are widely used, they mainly boost individual productivity rather than

In [223]:
print("AUTHOR:")
print(article_analysis.author)

print("\nTITLE:")
print(article_analysis.title)

print("\nRELEVANCE:")
print(article_analysis.relevance)

print("\nSUMMARY:")
print(article_analysis.summary)

AUTHOR:
MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

TITLE:
The GenAI Divide: State of AI in Business 2025

RELEVANCE:
This here article shines a light on the stark divide between high GenAI adoption and real transformation in business—figurin’ out what’s workin’ and what ain’t for folks in the AI world. For an AI professional, understanding these trends and patterns can help steer their ship toward more fruitful investments and implementations, keepin’ them ahead of the curve in a rapidly changin’ landscape.

SUMMARY:
In the report "The GenAI Divide: State of AI in Business 2025", the authors reveal a tall tale about the state of Generative AI (GenAI) in the business frontier. Despite a mighty investment between $30–40 billion in GenAI initiatives, a staggering 95% of organizations are seeing no returns—what they call the GenAI Divide. The report outlines that while tools like ChatGPT are widely used, they mainly boost individual productivity rather than

In [140]:
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("Total tokens:", response.usage.total_tokens)

Input tokens: 10928
Output tokens: 448
Total tokens: 11376


In [141]:
print(type(article_analysis))

<class '__main__.ArticleAnalysis'>


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [240]:
%pip install -U deepeval


python(42628) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


In [239]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [230]:
from deepeval.metrics import SummarizationMetric

summarization_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=[
        "how accurate is the summary in representing the main points?",
        "what level of detail is provided in the summary?",
        "how well does the summary capture the tone and style of the original article?",
        "how coherent and well-structured is the summary?",
        "how well does the summary convey the key insights and implications of the article?"
    ]
)


In [231]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=pdf_text,
    actual_output=article_analysis.summary
)

In [238]:
summarization_metric.measure(test_case)

Output()

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: <if avai**********************************nAI>. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [234]:
print("Gateway key present:", bool(client.default_headers.get("x-api-key")))

Gateway key present: True


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
